# VibeVoice Pipeline on Colab

Runs the full Video Generation Pipeline on Colab's free GPU (T4).


## 0. Clone repositories


In [ ]:
# ── (Recommended) Cache models to Google Drive → fast re-runs ────────
# Colab wipes everything between sessions, so normally it re-downloads the
# ~7GB voice model + whisper models EVERY run (the main reason it's slow).
# Pointing HF_HOME at Drive downloads them ONCE; every later run reuses them.
# Comment these lines out to skip Drive (you'll re-download each session).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'   # models persist here
    print("Model cache -> Drive: first run downloads, later runs reuse (much faster).")
except Exception as e:
    print("Drive not mounted; models will re-download each session:", e)

# ── GPU check ────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), (
    "No GPU! Top menu: Runtime -> Change runtime type -> T4 GPU, then re-run all cells. "
    "On CPU the model load exhausts RAM and Colab kills the process."
)
print("GPU:", torch.cuda.get_device_name(0))

# ── Model choice ─────────────────────────────────────────────
# "0.5B"  — fast streaming model, fixed .pt voices (free T4 OK)
# "1.5B"  — TRUE voice cloning from a reference WAV (free T4 OK, ~7GB)
# "Large" — the 7B-class model, best quality (needs Colab Pro A100/L4, ~20GB VRAM)
VIBEVOICE_MODEL = "1.5B"

import os
os.environ["VIBEVOICE_MODEL"] = VIBEVOICE_MODEL

!rm -rf audio
!git clone https://github.com/saitejatiru/audiowithvideoremotion.git audio
%cd audio
!rm -rf VibeVoice
if VIBEVOICE_MODEL == "0.5B":
    # microsoft repo has the streaming model code + .pt voices
    !git clone https://github.com/microsoft/VibeVoice.git
else:
    # community fork restores the non-streaming TTS code + reference WAV voices
    !git clone https://github.com/vibevoice-community/VibeVoice.git
import sys; sys.path.insert(0, ".")


## 1. Install dependencies
Installs requirements for TTS, Alignment, Storyboarding, Rendering, and Orchestration.


In [ ]:
!pip install -q fastapi "uvicorn[standard]" soundfile requests nemo-text-processing gradio tenacity "pandas<3.0.0" "huggingface-hub>=1.2.0"
!pip install -q -r align/requirements.txt
!pip install -q openai json-repair

# VibeVoice model package — REQUIRED, run_vibevoice imports `vibevoice`
!pip install -q -e VibeVoice

# Manim — real 2D animation for Physics/Maths 'animation' scenes (CPU render)
!sudo apt-get -qq install -y libcairo2-dev libpango1.0-dev ffmpeg >/dev/null 2>&1
!pip install -q manim

# Node.js + Remotion dependencies for Phase 4 rendering
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!cd video && npm install
!sudo apt-get install -yq gconf-service libasound2 libatk1.0-0 libc6 libcairo2 libcups2 libdbus-1-3 libexpat1 libfontconfig1 libgcc1 libgconf-2-4 libgdk-pixbuf2.0-0 libglib2.0-0 libgtk-3-0 libnspr4 libpango-1.0-0 libpangocairo-1.0-0 libstdc++6 libx11-6 libx11-xcb1 libxcb1 libxcomposite1 libxcursor1 libxdamage1 libxext6 libxfixes3 libxi6 libxrandr2 libxrender1 libxss1 libxtst6 ca-certificates fonts-liberation libappindicator1 libnss3 lsb-release xdg-utils wget

## 2. Set API Keys


In [ ]:
import os

# ── LLM provider: GROQ (free, reliable, verified working) ────
# Repo is PUBLIC — do NOT commit your key. Paste it here at runtime; it stays
# in the Colab session only. Your key was verified working with this model.
os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["LLM_API_KEY"]  = "<PASTE_YOUR_GROQ_KEY>"   # gsk_... from console.groq.com
os.environ["LLM_MODEL"]    = "llama-3.3-70b-versatile"
#
# Alternative — NVIDIA NIM (free credits are LIMITED, can 401 when exhausted):
#   base https://integrate.api.nvidia.com/v1, model meta/llama-3.3-70b-instruct

# ── Visuals: Pixabay (free) for illustrations + stock video ──
# Repo is PUBLIC — do NOT commit your key. Paste it here at runtime, or use
# Colab Secrets (🔑): os.environ["PIXABAY_API_KEY"] = userdata.get("PIXABAY_API_KEY")
os.environ["PIXABAY_API_KEY"] = "<PASTE_YOUR_PIXABAY_KEY_HERE>"
# os.environ["PIXABAY_VIDEO"] = "1"   # uncomment to use stock VIDEO b-roll
                                       # (motion, but keyword match is looser)

os.environ["GRADIO_SHARE"] = "0"

# ── Smoke test — both lines must say OK before you generate.
#    !! After changing ANY key here, RE-RUN cell 8 (app.py) too — a running app
#    keeps the OLD key, which causes a 401 at generate time. ──
if "<" in os.environ["LLM_API_KEY"]:
    print("!! LLM_API_KEY placeholder — storyboard will FAIL. Paste your Groq key.")
else:
    from openai import OpenAI
    _c = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
    try:
        _r = _c.chat.completions.create(
            model=os.environ["LLM_MODEL"],
            messages=[{"role": "user", "content": "Reply with the word: ok"}],
            max_tokens=10,
        )
        print("LLM OK:", os.environ["LLM_MODEL"], "->", _r.choices[0].message.content)
    except Exception as e:
        print("=" * 70)
        print("LLM BROKEN:", repr(e))
        if "401" in str(e) or "Unauthorized" in str(e):
            print("-> 401: key invalid / expired. Regenerate at console.groq.com.")
        else:
            print("   Check LLM_MODEL is spelled exactly right.")
        print("=" * 70)

# Pixabay check
if "<" in os.environ["PIXABAY_API_KEY"]:
    print("!! PIXABAY_API_KEY placeholder — scenes will have NO illustrations/video. Paste your key above.")
else:
    import requests
    try:
        _n = requests.get("https://pixabay.com/api/", params={
            "key": os.environ["PIXABAY_API_KEY"], "q": "human heart",
            "image_type": "illustration", "per_page": 3}, timeout=10
        ).json().get("totalHits", 0)
        print("PIXABAY OK — 'human heart' illustrations available:", _n)
    except Exception as e:
        print("PIXABAY BROKEN:", repr(e))


## 3. Launch the Full Gradio UI
This will expose the complete pipeline (TTS -> Align -> Storyboard -> Render -> Post-Process) through a public Gradio URL.


In [ ]:
%cd /content/audio
# Launch the app and open it via Colab's OWN port proxy — this does NOT depend
# on Gradio's share tunnel (the "Could not create share link" service that
# often fails). The UI renders inline below; app logs also stream below.
import os, socket, time, subprocess
os.environ["GRADIO_SHARE"] = "0"   # don't attempt the flaky share tunnel

log = open("/content/app.log", "w")
proc = subprocess.Popen(["python", "app.py"], stdout=log, stderr=subprocess.STDOUT)

print("Starting app (loading models)…")
up = False
for _ in range(180):  # up to ~6 min for heavy imports
    try:
        socket.create_connection(("127.0.0.1", 7860), timeout=1).close()
        up = True
        break
    except OSError:
        time.sleep(2)

if up:
    print("App is up — rendering the UI inline below (scroll down):")
    from google.colab import output
    # iframe = Colab's recommended, more reliable embed (window popups get blocked)
    output.serve_kernel_port_as_iframe(7860, height="800")
else:
    print("App did not come up in time — check the log below.")

# stream the app's logs here (pipeline progress + any errors)
!tail -f -n +1 /content/app.log
